# 04 — Deep Blue data: organ and lesion absorbed doses from three TIA maps

The absorbed dose is computed from three time-integrated activity (TIA) maps of the same patients, with the same dose engine, so that the differences between the doses come only from the TIA:

| TIA map | Spatial distribution | Kinetics |
|---|---|---|
| **MIM** | MIM voxel-wise fits (reference map of the dataset) | MIM voxel-wise fits |
| **voxel** | our voxel-wise uptake–washout fits on the `bin2` grid (notebook 03) | voxel-wise |
| **hybrid** | registered SPECTs (notebook 03), inside each structure | organ-level uptake–washout fits (notebook 02) |

The hybrid map is the classical compromise between organ-level and voxel-level dosimetry: the kinetics, which need several time points and are noisy at the voxel level, are taken from the organ fit; only the shape of the distribution inside the organ comes from the images. It is also the structure of the project model in its simplest form: kinetic parameters per region, spatial detail from imaging.

**Dose engines**, both based on the Lu-177 voxel S-value (VSV) kernel computed with GATE in water (4 mm voxels, $21^3$ voxels, $10^7$ decays, mGy per MBq·s in the source voxel):
1. **VSV convolution**: $D = \tilde A * K$, with an FFT convolution;
2. **local energy deposition (LED)**: all the energy that the kernel deposits within its 84 mm extent ($\sum K$, 0.158 MeV per decay) is deposited in the source voxel, $D = \tilde A \sum K$.

**Steps**
1. Load the TIA maps, the registered SPECTs, the organ fits and the kernel.
2. Build the hybrid TIA map.
3. Bring the three maps to the 4 mm grid of the kernel, conserving the total TIA.
4. Compute the dose maps with both engines.
5. Mean absorbed dose per organ and lesion; comparison between the three maps; dose-volume histograms.
6. Save.

**About this notebook.** Organ and lesion doses from three TIA maps with the 4 mm VSV kernel and local deposition.
- **Inputs:** nb02, nb03 outputs; `output/vsv/` kernel.
- **Outputs:** `data/processed/deepblue/<patient>/dose/`, `kinetics/organ_doses.csv`.
- **Run time:** minutes.
- **Environment:** .venv.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy.signal import fftconvolve

WS = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "paths.py").exists())
sys.path.insert(0, str(WS / "src"))
from paths import DATASET
PROC = WS / "data" / "processed" / "deepblue"
KIN = PROC / "kinetics"
PATIENTS = ["patient_4", "patient_6"]
SNMMI_LABEL = {"patient_4": "A", "patient_6": "B"}
MAPS = ["MIM", "voxel", "hybrid"]
ENGINES = ["VSV", "LED"]

## 1. Loading

From notebooks 01–03: the scan-1 SPECT (reference grid, 1.95 mm), the scan-1 masks, the MIM TIA map, our `bin2` voxel TIA map (3.9 mm), the native voxel TIA map (used only outside the structures of the hybrid map), the four SPECTs registered to scan 1, and the organ-level `uptake` fits of notebook 02.

In [ ]:
def read(path):
    return sitk.ReadImage(str(path))


def arr(image):
    return sitk.GetArrayFromImage(image)


def like(array, reference):
    image = sitk.GetImageFromArray(array)
    image.CopyInformation(reference)
    return image


def voxel_ml(image):
    return float(np.prod(image.GetSpacing())) / 1000.0


data = {}
for patient in PATIENTS:
    pdir, vdir = PROC / patient, PROC / patient / "voxel"
    timing = json.loads((pdir / "timing.json").read_text())
    scans = sorted(timing["t_h"], key=timing["t_h"].get)
    ref = read(pdir / "scan1" / "spect_bqml.nii.gz")
    data[patient] = {
        "ref": ref,
        "t_h": np.array([timing["t_h"][s] for s in scans]),
        "masks": {p.name.removesuffix(".nii.gz").split("_", 1)[1]: arr(read(p)) > 0
                  for p in sorted((pdir / "scan1" / "masks").glob("*.nii.gz"))},
        "aligned": [arr(read(vdir / f"spect_aligned_{s}.nii.gz")).astype(np.float64) for s in scans],
        "tia": {"MIM": read(pdir / "tia_mim_bqml_s.nii.gz"),
                "voxel": read(vdir / "tia_bqml_s_bin2.nii.gz"),
                "voxel_native": read(vdir / "tia_bqml_s_native.nii.gz")},
    }

LAM_PHYS = np.log(2) / (timing["half_life_s"] / 3600.0)
fits = pd.read_csv(KIN / "fits_all_models.csv")
fits = fits[fits.model == "uptake"].set_index(["patient", "structure"])

kernel = np.load(WS / "output" / "vsv" / "vsv_lu177_mGy_per_MBq_s.npy")
KERNEL_MM = 4.0
print("kernel", kernel.shape, f"centre {kernel[10, 10, 10]:.4f}, sum {kernel.sum():.4f} mGy/(MBq s)")
print({p: list(data[p]["masks"]) for p in PATIENTS})

## 2. Hybrid TIA map

Inside each structure $R$ (scan-1 masks; lesions take precedence over the liver, so the liver region is the healthy liver), the hybrid TIA is

$$\text{TIA}_\text{hyb}(\mathbf{x}) = \text{TIA}_R \sum_{k=1}^{4} c_k\, \frac{A_k(\mathbf{x})}{\sum_{\mathbf{y} \in R} A_k(\mathbf{y})\,V}, \qquad \sum_k c_k = 1,$$

where $\text{TIA}_R$ is the TIA of the organ-level uptake fit of notebook 02, $A_k$ is the registered SPECT of scan $k$, and $c_k$ is the fraction of the fitted organ curve that each scan represents: the time axis is split at the midpoints between scan times ($0 \to m_1 \to m_2 \to m_3 \to \infty$) and $c_k$ is the integral of the fitted $A_R(t)$ over the $k$-th interval divided by $\text{TIA}_R$. The early scans therefore shape the distribution of the uptake phase and the last scan that of the tail. By construction the sum of the hybrid map over $R$ equals $\text{TIA}_R$ exactly.

Organ-level fits exist for kidneys, spleen, healthy liver and lesions. Outside these structures the hybrid map uses the native voxel map of notebook 03.

In [ ]:
def fit_params(patient, structure):
    f = fits.loc[(patient, structure)]
    lw = np.log(2) / f.T_eff_h
    lu = np.log(2) / f.T_uptake_h
    a0 = f.TIA_MBq_h / (1 / lw - 1 / lu)
    return a0, lw, lu


def interval_integral(a0, lw, lu, t0, t1):
    prim = lambda t: a0 * (-np.exp(-lw * t) / lw + np.exp(-lu * t) / lu)
    return (0.0 if np.isinf(t1) else prim(t1)) - prim(t0)


hybrid_qc = []
for patient in PATIENTS:
    d = data[patient]
    t = d["t_h"]
    edges = np.concatenate([[0.0], (t[:-1] + t[1:]) / 2, [np.inf]])
    lesions = [n for n in d["masks"] if n.startswith("lesion")]
    any_lesion = np.any([d["masks"][n] for n in lesions], axis=0)
    regions = {n: d["masks"][n] for n in d["masks"] if n not in lesions + ["liver"]}
    regions["liver_healthy"] = d["masks"]["liver"] & ~any_lesion
    for n in lesions:
        regions[n] = d["masks"][n]
    for n in list(regions):
        if not n.startswith("lesion"):
            regions[n] = regions[n] & ~any_lesion
    hybrid = arr(d["tia"]["voxel_native"]).astype(np.float64)
    v = voxel_ml(d["ref"])
    for n, m in regions.items():
        a0, lw, lu = fit_params(patient, n)
        tia_r = fits.loc[(patient, n), "TIA_MBq_h"] * 1e6 * 3600.0
        c = np.array([interval_integral(a0, lw, lu, edges[k], edges[k + 1]) for k in range(len(t))])
        c /= c.sum()
        dist = sum(ck * a[m] / (a[m].sum() * v) for ck, a in zip(c, d["aligned"]))
        hybrid[m] = tia_r * dist
        hybrid_qc.append({"patient": patient, "structure": n, **{f"c_scan{k + 1}": ck for k, ck in enumerate(c)},
                          "sum_over_fit": hybrid[m].sum() * v / tia_r})
    d["tia"]["hybrid_native"] = like(hybrid.astype(np.float32), d["ref"])
    d["regions"] = regions
pd.DataFrame(hybrid_qc).set_index(["patient", "structure"]).round(3)

## 3. Common 4 mm grid

The kernel is defined on 4 mm voxels, so all TIA maps are brought to a 4 mm grid covering the scan-1 field of view:
- native maps (MIM, hybrid) are first averaged 2×2×2 to 3.9 mm, which conserves the activity exactly; the voxel map is already on this grid;
- the 3.9 mm maps are resampled to 4 mm by linear interpolation (a 2.4% change of voxel size);
- each map is rescaled by a single factor so that its total TIA equals the total on the 3.9 mm grid. The factors are reported and should be within a fraction of a percent of 1.

The dose maps are computed on the 4 mm grid and resampled back (linearly) to the native scan-1 grid, where the organ means are taken with the native masks. Activity concentrations are converted to cumulated activity per voxel, $\tilde A = \text{TIA} \cdot V_\text{voxel}$, in MBq·s.

In [ ]:
def grid_4mm(ref):
    size = [int(np.floor(ref.GetSize()[i] * ref.GetSpacing()[i] / KERNEL_MM)) for i in range(3)]
    out = sitk.Image(size, sitk.sitkFloat32)
    out.SetSpacing([KERNEL_MM] * 3)
    out.SetDirection(ref.GetDirection())
    corner = np.array(ref.TransformContinuousIndexToPhysicalPoint([-0.5, -0.5, -0.5]))
    direction = np.array(ref.GetDirection()).reshape(3, 3)
    out.SetOrigin((corner + direction @ np.full(3, KERNEL_MM / 2)).tolist())
    return out


def to_4mm(tia_39, grid):
    resampled = sitk.Resample(sitk.Cast(tia_39, sitk.sitkFloat32), grid, sitk.Transform(), sitk.sitkLinear, 0.0)
    total_39 = float(arr(tia_39).astype(np.float64).sum()) * voxel_ml(tia_39)
    total_4 = float(arr(resampled).astype(np.float64).sum()) * voxel_ml(resampled)
    return arr(resampled).astype(np.float64) * total_39 / total_4, total_39 / total_4


grid_qc = []
for patient in PATIENTS:
    d = data[patient]
    d["grid"] = grid_4mm(d["ref"])
    sources = {"MIM": sitk.BinShrink(sitk.Cast(d["tia"]["MIM"], sitk.sitkFloat32), [2, 2, 2]),
               "voxel": d["tia"]["voxel"],
               "hybrid": sitk.BinShrink(d["tia"]["hybrid_native"], [2, 2, 2])}
    d["tia_4mm"] = {}
    for name, img in sources.items():
        d["tia_4mm"][name], factor = to_4mm(img, d["grid"])
        grid_qc.append({"patient": patient, "map": name, "grid_size": d["grid"].GetSize(), "rescale_factor": factor,
                        "total_TIA_MBq_h": d["tia_4mm"][name].sum() * KERNEL_MM**3 / 1000 / 3.6e9})
pd.DataFrame(grid_qc).set_index(["patient", "map"]).round(4)

## 4. Dose maps

$$D_\text{VSV}(\mathbf{x}) = \sum_{\mathbf{y}} \tilde A(\mathbf{y})\, K(\mathbf{x} - \mathbf{y}), \qquad D_\text{LED}(\mathbf{x}) = \tilde A(\mathbf{x}) \sum K$$

with $\tilde A$ in MBq·s and $K$ in mGy/(MBq·s); doses are reported in Gy. The kernel is computed in water: no density correction is applied, which is adequate for soft-tissue organs and lesions in the abdomen. Energy deposited farther than 40 mm from the source voxel (part of the photon contribution) is not included in either engine.

In [ ]:
voxel_mbq_s = lambda tia: tia * KERNEL_MM**3 / 1000 / 1e6
for patient in PATIENTS:
    d = data[patient]
    d["dose_4mm"] = {}
    for name in MAPS:
        a = voxel_mbq_s(d["tia_4mm"][name])
        d["dose_4mm"][(name, "VSV")] = fftconvolve(a, kernel, mode="same") / 1000.0
        d["dose_4mm"][(name, "LED")] = a * kernel.sum() / 1000.0
    d["dose_native"] = {k: arr(sitk.Resample(like(v.astype(np.float32), d["grid"]), d["ref"], sitk.Transform(),
                                             sitk.sitkLinear, 0.0)).astype(np.float64)
                        for k, v in d["dose_4mm"].items()}
    print(patient, "max VSV dose [Gy]:", {n: round(float(d["dose_4mm"][(n, "VSV")].max()), 1) for n in MAPS})

Coronal slice through the organs: VSV dose from the three TIA maps, with the scan-1 contours.

In [ ]:
for patient in PATIENTS:
    d = data[patient]
    allmask = np.any(list(d["masks"].values()), axis=0)
    y = int(np.argwhere(allmask).mean(axis=0)[1])
    vmax = np.percentile(d["dose_native"][("MIM", "VSV")][allmask], 99)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, name in zip(axes, MAPS):
        im = ax.imshow(d["dose_native"][(name, "VSV")][::-1, y, :], cmap="inferno", vmin=0, vmax=vmax, aspect="auto")
        ax.contour(allmask[::-1, y, :], [0.5], colors="cyan", linewidths=0.5)
        ax.set_title(f"{name} TIA, VSV dose")
        ax.axis("off")
    fig.colorbar(im, ax=axes, fraction=0.02, label="Gy")
    fig.suptitle(f"{patient} (SNMMI patient {SNMMI_LABEL[patient]})")

## 5. Organ and lesion doses

Mean absorbed dose over each structure, for the three TIA maps and the two engines. Structures are the scan-1 masks; the liver is reported as healthy liver (lesions excluded), as in the SNMMI challenge; `kidneys` is the mean over both kidneys (mass-weighted, since the masks are pooled). The TIA of each structure on the native grid is reported next to the dose, so that dose ratios can be compared with TIA ratios.

In [ ]:
rows = []
for patient in PATIENTS:
    d = data[patient]
    structures = dict(d["regions"])
    if {"ltkidney", "rtkidney"} <= set(structures):
        structures["kidneys"] = structures["ltkidney"] | structures["rtkidney"]
    tia_native = {"MIM": arr(d["tia"]["MIM"]).astype(np.float64), "hybrid": arr(d["tia"]["hybrid_native"]).astype(np.float64)}
    v = voxel_ml(d["ref"])
    for n, m in structures.items():
        row = {"patient": patient, "snmmi": SNMMI_LABEL[patient], "structure": n, "volume_ml": m.sum() * v}
        for name in MAPS:
            for engine in ENGINES:
                row[f"D_{name}_{engine}_Gy"] = d["dose_native"][(name, engine)][m].mean()
        rows.append(row)
doses = pd.DataFrame(rows).set_index(["patient", "structure"])
for name in ["voxel", "hybrid"]:
    doses[f"{name}_over_MIM_VSV"] = doses[f"D_{name}_VSV_Gy"] / doses["D_MIM_VSV_Gy"]
for name in MAPS:
    doses[f"VSV_over_LED_{name}"] = doses[f"D_{name}_VSV_Gy"] / doses[f"D_{name}_LED_Gy"]
doses.to_csv(KIN / "organ_doses.csv")
doses[["volume_ml"] + [f"D_{n}_VSV_Gy" for n in MAPS] + ["voxel_over_MIM_VSV", "hybrid_over_MIM_VSV"]].round(3)

Effect of the dose engine: VSV dose divided by LED dose for each map. For organs much larger than the electron range the two engines should agree within a few percent; for small lesions the convolution spreads the energy outside the lesion (and brings in energy from the surroundings), so the ratio departs from 1.

In [ ]:
doses[[f"VSV_over_LED_{n}" for n in MAPS]].round(3)

Cumulative dose-volume histograms (VSV) of the kidneys, the healthy liver, the spleen and the lesions, for the three TIA maps (dose axis up to 1.2 times the 99.5th percentile of the MIM dose). The mean doses can agree while the histograms differ: the voxel and MIM maps carry the voxel-level noise and the registration errors, the hybrid map only the spatial pattern of the images.

In [ ]:
colors = {"MIM": "black", "voxel": "tab:red", "hybrid": "tab:blue"}
for patient in PATIENTS:
    d = data[patient]
    names = [n for n in ["ltkidney", "rtkidney", "liver_healthy", "spleen"] if n in d["regions"]] + \
            sorted(n for n in d["regions"] if n.startswith("lesion"))
    ncol = 4
    nrow = int(np.ceil(len(names) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3.2 * nrow))
    for ax, n in zip(axes.ravel(), names):
        m = d["regions"][n]
        for name in MAPS:
            dose = np.sort(d["dose_native"][(name, "VSV")][m])
            ax.plot(dose, 1 - np.arange(dose.size) / dose.size, color=colors[name], label=name)
        ax.set_xlim(0, 1.2 * np.percentile(d["dose_native"][("MIM", "VSV")][m], 99.5))
        ax.set_title(n, fontsize=10)
        ax.set_xlabel("Dose [Gy]")
        ax.set_ylabel("Volume fraction")
    for ax in axes.ravel()[len(names):]:
        ax.axis("off")
    axes.ravel()[0].legend(fontsize=8)
    fig.suptitle(f"{patient}: cumulative DVH, VSV")
    fig.tight_layout()

### Findings

- **Conversion to the 4 mm grid** changes the total TIA of each map by less than 0.1% (rescale factors 0.999–1.000). The total TIA of the voxel map is 89% (patient 4) and 96% (patient 6) of MIM; the hybrid map 93% and 95%. These totals include the background, where the maps differ most (the voxel map is limited to the fitted body).
- **Hybrid vs MIM** (mean dose, VSV): kidneys 0.97–1.03 (pooled kidneys 1.00 for patient 4 and 0.98 for patient 6), spleen 0.98, healthy liver 0.96 and 1.05, lesions 0.98–1.10, except lesion 2 (1.32 for patient 4, 1.19 for patient 6).
- **Voxel vs MIM**: kidneys 0.95–1.04, spleen 0.94, healthy liver **0.83** (patient 4) and 1.04 (patient 6), lesions 0.99–1.09, except lesion 2 (1.24 and 1.19).
- **Healthy liver of patient 4: the voxel fits are biased low on noisy curves.** On the native grid the sum of the voxel-fit TIAs in the healthy liver is 25.4 GBq·h, against 29.0 GBq·h for the organ-level fit of notebook 02, 30.0 GBq·h for a non-parametric integration of the summed liver curve and 30.4 GBq·h for MIM, i.e. 12–16% low; for patient 6, whose liver curves are much less noisy (median RMS residual 0.12 against 0.30), the deficit is 1.5%. Fitting a non-linear model with relative weights to noisy points gives more weight to the points that the noise has pushed down, so the fitted curve is biased low: in a simulation with independent Gaussian noise the bias of the uptake model is −1% at 10% noise and −4% at 30% noise. The correlated, non-Gaussian noise of these reconstructions produces a larger bias. This is the main argument for the hybrid map in noisy regions: the kinetics are fitted where the signal is high (the organ curve), and the images contribute only the spatial pattern.
- **Lesion 2** of both patients is 19–32% above MIM with both of our maps, consistent with the TIA differences found at the organ level (extrapolation with $\lambda_\text{bio}$ on its bound for patient 4, uptake phase for patient 6) and with the different alignment of a 4 mL lesion; the DVH of MIM for lesion 2 of patient 4 extends to much lower doses, which points to a partial misalignment of the lesion in the MIM map.
- **Dose engine.** VSV/LED is 0.98 for kidneys, spleen and large lesions, 1.00 for the liver and 0.95–0.97 for lesions below 25 mL: in a small lesion part of the electron energy is deposited outside the lesion, which LED ignores. For organ means LED is within 2% of the convolution; the difference between the two engines is much smaller than the differences between the TIA maps.
- **DVHs.** For kidneys, spleen and large lesions the three maps give similar histograms. The healthy liver of patient 4 has a long high-dose tail from the voxels next to lesion 1 (cross-fire and spill-out of a 116 mL lesion with a mean dose of about 20 Gy), present in all three maps.

## 6. Saving

In `data/processed/deepblue/<patient>/dose/` (4 mm grid): `tia_<map>_4mm.nii.gz` (Bq·s/mL) and `dose_<map>_<engine>_4mm.nii.gz` (Gy) for the three maps and two engines; `tia_hybrid_native.nii.gz` (Bq·s/mL, scan-1 grid). In `data/processed/deepblue/kinetics/`: `organ_doses.csv`.

In [ ]:
for patient in PATIENTS:
    d = data[patient]
    ddir = PROC / patient / "dose"
    ddir.mkdir(parents=True, exist_ok=True)
    sitk.WriteImage(d["tia"]["hybrid_native"], str(ddir / "tia_hybrid_native.nii.gz"))
    for name in MAPS:
        sitk.WriteImage(like(d["tia_4mm"][name].astype(np.float32), d["grid"]), str(ddir / f"tia_{name}_4mm.nii.gz"))
        for engine in ENGINES:
            sitk.WriteImage(like(d["dose_4mm"][(name, engine)].astype(np.float32), d["grid"]),
                            str(ddir / f"dose_{name}_{engine}_4mm.nii.gz"))
    print("saved", ddir)